### Adaptive RAG
Adaptive RAG is framework that dynamically adjusts its strategy for handling queries based on their complexity. It's like having a smart assistant that knows when to dig deep for information and when to provide a simple answer. Instead of using a single, rigid approach, Adpative RAG chooses the most appropriate retrieval method for each query , balancing speed and accuracy.

Adaptive RAG is a strategy for that unites (1) Query Ananlysis with (2) active/self-corretive RAG.

In [7]:
import os
from dotenv import load_dotenv
load_dotenv() ## Loading all environment variable

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

In [8]:
import truststore
truststore.inject_into_ssl()

In [9]:
## Build Index

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

### from lanchain_cohere import CorhereEmbeddings

embd = OpenAIEmbeddings()

#Docs to index
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

#split
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=500, chunk_overlap=0
    )

docs_splits = text_splitter.split_documents(docs_list)

#add to vector store
vector_store = FAISS.from_documents(
    documents=docs_splits, 
    embedding=OpenAIEmbeddings())

retriever = vector_store.as_retriever()

In [10]:
### Router 

from typing import Literal

from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

from pydantic import BaseModel, Field

#Data model
class RouteQuery(BaseModel):
    """Route a user query to the most relevant data source."""

    datasource : Literal["vector_store", "web_search"] = Field(
        ...,
        description="Given a user question to route it to web search or vector store"
    )


# LLM with function call

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)
structured_llm_router = llm.with_structured_output(RouteQuery)

#Promot

System = """ You are an expert at routing a user question to a vector store or web search.
        The vectorstore contains documents related to agents, prompt engineering, and adversarial attacks.
        Use the vector store for questions on these topics. Otherwise, use web search. """


route_prompt = ChatPromptTemplate.from_messages([
    ("system", System),
    ("human", "{question}"),
])

question_router = route_prompt | structured_llm_router


print(
    question_router.invoke({
        "question": "Who won the cricket world world cup 2023 ?"
    })
)




datasource='web_search'


In [11]:
print(question_router.invoke({
    "question": "What are the types of agent memory ?"
}))

datasource='vector_store'


In [21]:
### Retrieval Grader

#Data model
class GradeDocument(BaseModel):
    """Binary score for relecance check on retrieved documents"""

    binary_score: str = Field(
        description="Document are relevant to the question, 'yes' or 'no'"    
    )

#LLM with Function call
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
structured_llm_grader = llm.with_structured_output(GradeDocument)

#Prompt
system = """ You are grader assessing relevance of a retrieved document to a user question. \n
If the document contains keyword(s) or semantic meaning releated to the user question, grade it as relevant. \n
It does not need to be stringent test. The goal is to filter out errorneous retrievals. \n
Give a binay score 'yes' or 'no' score to indicate whether the document is relevant to the question."""

grade_prmopt = ChatPromptTemplate.from_messages(
    [
        ("system" , system),
        ("human"  , "Retrieved document: \n\n {document} User Question: {question}")
    ]
)

retrieval_grader = grade_prmopt | structured_llm_grader
question = "agent memory"
docs = retriever.invoke(question)
docs_txt = docs[1].page_content
print(retrieval_grader.invoke({"question" : question, "document" : docs_txt}))


binary_score='yes'


In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langsmith import Client

#prompt parser for structured output
client = Client()
#Prompt
prompt = client.pull_prompt("rlm/rag-prompt", 
                dangerously_pull_public_prompt=True,)

#LLM
llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)

#Post-pocessing
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

#Chain
rag_chain = prompt | llm | StrOutputParser()

#Run
generation = rag_chain.invoke({"context" : docs, "question" : question})
print(generation)


Agent memory in LLM-powered autonomous systems consists of short-term and long-term memory. Short-term memory utilizes in-context learning for immediate tasks, while long-term memory allows agents to retain and recall information over extended periods, often using external storage for efficient retrieval. This dual memory system enhances the agent's ability to learn from past experiences and improve future actions.


In [ ]:
### Hallucination Grader

# Data Model
class GradeHallucinations(BaseModel):
    """Binary score for hallucination present in generation answer."""

    binary_score : str = Field(
        description=" Answer is grounded in the fact, 'yes' or 'No'"
    )

#LLM with function call
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
structured_llm_grader = llm.with_structured_output(GradeHallucinations)

# Prompt
system = """You are grader assessing whether whether an LLM generation is grounded in / support by a set of retrieved facts. \n
Give a binary score 'yes' or 'no'. 'Yes' means that the wnser is grounded in / supported by the set of facts."""

hallucination_prompt = ChatPromptTemplate.from_messages([
    ("system", system),
    ("human", "Set of facts: \n\n {documents} \n\n LLM generation: {generation}")
]
)

hallucination_grader = hallucination_prompt| structured_llm_grader
hallucination_grader.invoke({"documents" : docs, "generation" : generation})


GradeHallucinations(binary_score='yes')

In [27]:
### Answer Grader

# Data Model
class GradeAnswer(BaseModel):
    """Binary score to assess answer addressing question correctly."""

    binary_score : str = Field(
        description=" Answer addresses the question correctly, 'yes' or 'No'"
    )

#LLM with function call
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
structured_llm_grader = llm.with_structured_output(GradeAnswer)

# Prompt
system = """You are grader assessing whether an LLM generation correctly addresses /resolve the question. \n
Give a binary score 'yes' or 'no'. 'Yes' means that the answer addresses the question correctly."""

answer_prompt = ChatPromptTemplate.from_messages([
    ("system", system),
    ("human", "Question: {question} \n\n LLM generation: {generation}")
]
)

answer_grader = answer_prompt | structured_llm_grader
answer_grader.invoke({"question" : question, "generation" : generation})

GradeAnswer(binary_score='yes')

In [28]:
### Question Re-writer

#LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Prompt
system = """You are a question re-writer that converts an input question to a better version that is optimized \n
for vector retrieval. Look at the input and try to reason about the underlying semantic intent / meaning."""

question_rewriter_prompt = ChatPromptTemplate.from_messages([
    ("system", system),
    ("human", "Here is the initial question: \n\n {question} \n Formulate an improved question.")
]
)

question_rewriter = question_rewriter_prompt | llm | StrOutputParser()
question_rewriter.invoke({"question" : question})

'What are the key concepts and functions of agent memory in artificial intelligence?'

In [ ]:
### Search

from langchain_community.tools.tavily_search import TavilySearchResults

web_search_tool = TavilySearchResults(K=3)


In [30]:
from typing import List
from typing_extensions import TypedDict

class GraphState(TypedDict):
    """
    
    Represents the state of the graph in the Adaptive RAG system.
    
    Attributes:
        question: question
        generation : LLM generation
        documents : List of documents

    """

    question: str
    generation: str
    documents: List[str]
    

Failed to refresh cache entry rlm/rag-prompt: Connection error caused failure to GET /commits/rlm/rag-prompt/latest in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /commits/rlm/rag-prompt/latest (Caused by NameResolutionError("HTTPSConnection(host=\'api.smith.langchain.com\', port=443): Failed to resolve \'api.smith.langchain.com\' ([Errno 11001] getaddrinfo failed)"))'))
Content-Length: None
API Key: 
Failed to refresh cache entry rlm/rag-prompt: Connection error caused failure to GET /commits/rlm/rag-prompt/latest in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /commits/rlm/rag-prompt/latest (Caused by NameResolutionError("HTTPSConnection(host=\'api.smith.langchain.com\', port=443): Failed to resolve \'api.smith.